In [1]:
import os
import sys
import torch
from torch import nn
sys.path.append(os.path.dirname(os.getcwd()))

In [2]:
from hydra import compose, core, initialize
from omegaconf import DictConfig, OmegaConf
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from src.data.data_module import EdaccDataModule
from transformers.models.whisper.modeling_whisper import WhisperConfig
from src.models.accent_token.model import WhisperWithAccentToken
from src.data.data_module import EdaccDataModule

In [3]:
with initialize(config_path="../configs", version_base=None):
    cfg = compose(config_name="baseline_eval.yaml", overrides=["+data.subset_mode=True"])


In [4]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base.en")

In [6]:
model.config.output_hidden_states

False

In [4]:
data = EdaccDataModule(**cfg.data)
data.prepare_data()
data.setup("fit")

for batch in data.train_dataloader():
    # print(batch)
    break

batch.keys()

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['input_features', 'attention_mask', 'labels', 'decoder_attention_mask', 'accent_id'])

In [5]:
# model = WhisperWithAccentToken.from_pretrained("openai/whisper-base.en")
processor = WhisperProcessor.from_pretrained("openai/whisper-base.en")

In [6]:
processor.tokenizer.batch_decode(batch["labels"])

['<|startoftranscript|><|notimestamps|>the cabbage right sort of<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>

In [6]:
processor.tokenizer.decode(processor.tokenizer("sasa")["input_ids"])

'<|startoftranscript|><|notimestamps|>sasa<|endoftext|>'

In [13]:
prompt_ids = processor.tokenizer.get_decoder_prompt_ids(language="en", task="transcribe")
# prompt_ids = [i[1] for i in prompt_ids]
# prompt_ids = torch.tensor(prompt_ids).unsqueeze(0).repeat(8, 1)

In [14]:
processor.tokenizer.batch_decode([i[1] for i in prompt_ids])

['<|en|>', '<|transcribe|>', '<|notimestamps|>']

In [13]:
processor.tokenizer.decode(processor.tokenizer.pad_token_id)

'<|endoftext|>'

In [11]:
processor.tokenizer.batch_decode(batch["labels"])

['<|startoftranscript|><|notimestamps|>of course<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|e

In [18]:
batch["accent_id"].unsqueeze(1)

tensor([[0],
        [1],
        [1],
        [0],
        [0],
        [0],
        [1],
        [1]])

In [30]:
a = model.generate(
    # decoder_input_ids=torch.randint(0, 100, (8, 1)),
    # forced_decoder_ids=prompt_ids,
    language="en",
    task="transcribe",
    input_features=batch["input_features"],
    attention_mask=batch["attention_mask"]
)

ValueError: Cannot specify `task` or `language` for an English-only model. If the model is intended to be multilingual, pass `is_multilingual=True` to generate, or update the generation config.

In [ ]:
processor.tokenizer.add_special_tokens(
    {"additional_special_tokens": [f"<|accent_{i}|>" for i in range(11)]}
)

In [29]:
processor.tokenizer.batch_decode(a)

[" Yeah, the closest thing we have is something called the sarmadhez, but it's not like a dumping. It's like wrapped in those like banana leaves, you know?<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>

In [ ]:
processor.tokenizer.batch_decode(a)